In [1]:
# conda activate anndata

import os
import sys
import anndata as ad

sys.path.append("/mnt/lareaulab/reliscu/code")

from junction2psi import *

In [ ]:
adata = ad.read_h5ad("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/hahn_2023/cortex/hahn_2023_cortex_STAR_SJ_counts.h5ad")

In [4]:
adata.shape

(166, 157406)

In [5]:
SJ_counts_table = pd.DataFrame(adata.X.T, columns=adata.obs_names, index=adata.var_names)

In [6]:
SJ_counts_table.shape

(157406, 166)

In [7]:
events_i1 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I1' in x])
events_i2 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I2' in x])
events_se = pd.Index([x[:-3] for x in SJ_counts_table.index if '_SE' in x])

events = events_i1.intersection(events_i2).intersection(events_se)
i1_events = [x + '_I1' for x in events]
I1_table = SJ_counts_table.loc[i1_events]
I1_table.index = events

i2_events = [x + '_I2' for x in events]
I2_table = SJ_counts_table.loc[i2_events]
I2_table.index = events

se_events = [x + '_SE' for x in events]
SE_table = SJ_counts_table.loc[se_events]
SE_table.index = events

I1_filt = I1_table.index[I1_table.sum(axis=1) > 0]
I2_filt = I2_table.index[I2_table.sum(axis=1) > 0]
SE_filt = SE_table.index[SE_table.sum(axis=1) > 0]
filtered_events = I1_filt.intersection(I2_filt).intersection(SE_filt)

I1_table = I1_table.loc[filtered_events]
I2_table = I2_table.loc[filtered_events]
SE_table = SE_table.loc[filtered_events]

psi = ((I1_table + I2_table) /(2*SE_table + I1_table + I2_table)).fillna(0)
reads = SE_table + I1_table + I2_table

In [8]:
psi.shape[0]

15116

In [9]:
psi.head()

,SRR21355739,SRR21355744,SRR21355733,SRR21355374,SRR21355671,SRR21355241,SRR21354990,SRR21355346,SRR21355701,SRR21355408,...,SRR21355376,SRR21355673,SRR21354992,SRR21355243,SRR21354998,SRR21354976,SRR21355344,SRR21355249,SRR21355703,SRR21355697
ENSMUSG00000025902_ProteinCoding_1,0.636364,1.0,1.0,1.0,0.000000,1.000000,1.000000,0.2,1.0,0.909091,...,1.000000,0.0,0.0,0.428571,0.8,1.0,1.000000,0.800000,1.0,1.0
ENSMUSG00000025902_ProteinCoding_2,0.636364,1.0,1.0,1.0,0.000000,1.000000,1.000000,0.2,1.0,0.909091,...,1.000000,0.0,0.0,0.428571,0.8,1.0,1.000000,0.777778,1.0,1.0
ENSMUSG00000025902_ProteinCoding_3,0.600000,1.0,1.0,0.0,0.000000,1.000000,0.000000,1.0,0.0,0.500000,...,1.000000,0.0,0.0,1.000000,1.0,0.0,1.000000,1.000000,1.0,1.0
ENSMUSG00000025902_ProteinCoding_4,0.666667,1.0,1.0,0.0,0.000000,1.000000,1.000000,1.0,0.0,0.000000,...,1.000000,0.0,0.0,1.000000,1.0,0.0,1.000000,1.000000,1.0,1.0
ENSMUSG00000033845_ProteinCoding_1,1.000000,1.0,1.0,1.0,0.967742,0.990148,0.989418,1.0,1.0,1.000000,...,0.979094,1.0,1.0,0.977778,1.0,1.0,0.990566,0.971292,1.0,1.0


In [ ]:
psi.to_csv(f"data/hahn_2023_cortex_STAR_exon_PSI.csv")
reads.to_csv(f"data/hahn_2023_cortex_STAR_exon_counts.csv")